# 🚀 AnimeKu Studio — 100% Free Video Ingestion Engine (Google Colab)
Notebook ini digunakan untuk mengunduh episode anime dari Nyaa.si, mengekstrak Subtitle Indonesia resmi, mengonversi Multi-Resolusi HLS `fMP4 Single-File` (1080p, 720p, 480p) dengan GPU gratis, dan mengunggahnya langsung ke **Cloudflare R2** serta mendaftarkannya ke database **Cloudflare D1** tanpa memakan kuota internet atau komputasi pribadi Anda.

> **Tips GPU Gratis:** Buka menu **Runtime** -> **Change runtime type** -> Pilih **T4 GPU** untuk akselerasi rendering hardware super cepat dari Google!

In [ ]:
#@title 📦 1. Persiapan Environment & Instalasi Tools (1-Click Run) { display-mode: "form" }
import os

print("⏳ [1/3] Menginstal aria2c, ffmpeg, rclone, dan Node.js 20...")
!apt-get update -qq > /dev/null
!apt-get install -y -qq aria2 ffmpeg rclone curl ca-certificates > /dev/null

# Setup Node.js 20 jika belum ada
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
!apt-get install -y -qq nodejs > /dev/null

print("⏳ [2/3] Menyiapkan repositori runner AnimeKu Worker...")
if not os.path.exists("/content/animeku-worker"):
    !git clone --depth 1 https://github.com/MrHyzee/animeku-worker.git /content/animeku-worker
else:
    %cd /content/animeku-worker
    !git pull

os.chdir("/content/animeku-worker")

print("⏳ [3/3] Menginstal dependencies Node.js...")
!npm install --no-audit --no-fund > /dev/null 2>&1

print("✅ Setup Selesai! Environment Google Colab siap mengeksekusi pipeline.")

In [ ]:
#@title ⚙️ 2. Parameter Konten & Ingestion Form { display-mode: "form" }
import os

ANIME_SLUG = "solo-leveling" #@param {type:"string"}
EPISODE_INPUT = "3" #@param {type:"string"} # Contoh: '3' untuk single, atau '1-12' untuk 1 season batch
MODE = "hls" #@param ["hls", "direct"]
STORAGE_DRIVER = "r2" #@param ["r2", "hetzner"]
MAGNET_OR_URL = "" #@param {type:"string"} # Opsional: Link magnet Nyaa. Kosongkan agar otomatis dicari oleh sistem!

# Kredensial Cloudflare (Otomatis diambil dari Colab Secrets 🔑 atau ketik di form bawah)
try:
    from google.colab import userdata
    CF_API_TOKEN = userdata.get('CF_API_TOKEN')
    R2_ACCESS_KEY = userdata.get('R2_ACCESS_KEY')
    R2_SECRET_KEY = userdata.get('R2_SECRET_KEY')
except Exception:
    CF_API_TOKEN = None
    R2_ACCESS_KEY = None
    R2_SECRET_KEY = None

CF_ACCOUNT_ID = "5b5d0b730a79c86cf659a0d01f86ba9f" #@param {type:"string"}
CF_DATABASE_ID = "b51144f3-8050-4349-815c-ecb3402510f7" #@param {type:"string"}
CF_API_TOKEN = CF_API_TOKEN or "" #@param {type:"string"}
R2_ACCESS_KEY = R2_ACCESS_KEY or "" #@param {type:"string"}
R2_SECRET_KEY = R2_SECRET_KEY or "" #@param {type:"string"}
R2_BUCKET_NAME = "animekuapp-media" #@param {type:"string"}

os.environ["CF_ACCOUNT_ID"] = CF_ACCOUNT_ID
os.environ["CF_DATABASE_ID"] = CF_DATABASE_ID
os.environ["CF_API_TOKEN"] = CF_API_TOKEN
os.environ["R2_ACCESS_KEY"] = R2_ACCESS_KEY
os.environ["R2_SECRET_KEY"] = R2_SECRET_KEY
os.environ["R2_BUCKET_NAME"] = R2_BUCKET_NAME
os.environ["STORAGE_DRIVER"] = STORAGE_DRIVER

print(f"🎯 Target Ingestion:")
print(f"   - Slug Anime : {ANIME_SLUG}")
print(f"   - Episode    : {EPISODE_INPUT}")
print(f"   - Mode       : {MODE.upper()} ({'Multi-Res HLS fMP4' if MODE == 'hls' else 'Direct MKV/MP4'})")
print(f"   - Storage    : {STORAGE_DRIVER.upper()}")

In [ ]:
#@title 🚀 3. Jalankan Eksekusi Ingestion { display-mode: "form" }
import subprocess

if "-" in EPISODE_INPUT:
    cmd = f"node scripts/ingest.js batch-season --anime '{ANIME_SLUG}' --range '{EPISODE_INPUT}' --mode '{MODE}'"
elif MAGNET_OR_URL.strip():
    cmd = f"node scripts/ingest.js auto-ingest --anime '{ANIME_SLUG}' --ep '{EPISODE_INPUT}' --magnet '{MAGNET_OR_URL.strip()}' --mode '{MODE}'"
else:
    cmd = f"node scripts/ingest.js auto-ingest --anime '{ANIME_SLUG}' --ep '{EPISODE_INPUT}' --mode '{MODE}'"

print(f"⚡ Menjalankan: {cmd}\n")
!{cmd}

print("\n🎉 Ingestion selesai! Seluruh episode telah terunggah ke Storage dan aktif di database D1 AnimeKu.")